In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from ugdatalab.models.galaxy_zoo import GalaxyZooDataset
from ugdatalab.models.galaxy_zoo.constants import (
    N_LABELS,
    LABEL_COLUMNS,
    LABEL_DESCRIPTIVE,
    MERGER_LABEL,
    PROTOTYPE_LABELS,
)
from ugdatalab.methods.cnn import predict_cnn, rmse_loss
from ugdatalab.methods.architectures import build_resnet18, build_custom_cnn

import plotters

# Galaxy Image Classification — Evaluation

This notebook covers the final analysis tasks:
1. **Task 23** — Compare validation loss across three models
2. **Task 24** — True vs predicted scatter for all 37 labels
3. **Task 25** — Top-5 extreme images for 7 specific labels
4. **Task 26** — Merger fraction on test images

In [ ]:
# Load data
img_data = np.load("galaxy_zoo_images.npz")
images = img_data["images"]
galaxy_ids = img_data["galaxy_ids"]
label_data = np.load("galaxy_zoo_labels.npz")
labels = label_data["labels"]
split_data = np.load("split_indices.npz")
train_idx, val_idx = split_data["train_idx"], split_data["val_idx"]

val_images = images[val_idx]
val_labels = labels[val_idx]
val_galaxy_ids = galaxy_ids[val_idx]
TARGET_SIZE = images.shape[1]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load all result files
resnet_data = np.load("resnet_result.npz")
custom_data = np.load("custom_result.npz", allow_pickle=True)
aug_data = np.load("augmented_result.npz")

## Task 23 — Model Comparison

In [ ]:
ax = plotters.plot_model_comparison(
    names=["Custom CNN", "ResNet-18", "ResNet-18 + Aug + LR"],
    val_losses_list=[
        custom_data["val_losses"],
        resnet_data["val_losses"],
        aug_data["val_losses"],
    ],
)
plt.show()

print(f"Custom CNN best val RMSE:       {float(custom_data['best_val_loss']):.4f}")
print(f"ResNet-18 best val RMSE:        {float(resnet_data['best_val_loss']):.4f}")
print(f"ResNet-18 + Aug + LR best RMSE: {float(aug_data['best_val_loss']):.4f}")

## Task 24 — True vs Predicted Scatter

We run the best model (augmented ResNet-18) on the validation set and make scatter plots comparing true vs predicted label values for all 37 labels.

In [ ]:
# Load best model and predict on validation set
best_model = build_resnet18(n_labels=N_LABELS, input_size=TARGET_SIZE)
best_model.load_state_dict(torch.load("resnet18_augmented.pt", weights_only=True))

val_ds = GalaxyZooDataset(val_images, val_labels, transform=None)
pred_labels = predict_cnn(best_model, val_ds, DEVICE, batch_size=64)

label_desc_list = [LABEL_DESCRIPTIVE[col] for col in LABEL_COLUMNS]
axes = plotters.plot_label_scatter(val_labels, pred_labels, label_desc_list)
plt.show()

## Task 25 — Top-5 Extreme Images

For 7 specific labels, we plot the 5 validation-set images with the highest probability, both by actual label and by model prediction. This reveals whether the model's most confident predictions match genuinely extreme morphologies.

Labels: (1) Smooth, (2) Star/Artifact, (3) Edge-on disk, (4) Odd: Ring, (5) Odd: Lens/Arc, (6) Spiral: 2 arms, (7) Odd: Merger.

In [ ]:
for proto_label in PROTOTYPE_LABELS:
    label_idx = LABEL_COLUMNS.index(proto_label)
    label_name = LABEL_DESCRIPTIVE[proto_label]

    axes = plotters.plot_top5_images(
        val_images,
        val_labels[:, label_idx],
        pred_labels[:, label_idx],
        label_idx,
        label_name,
        val_galaxy_ids,
    )
    plt.show()

## Task 26 — Merger Fraction

We run the trained model on the test image set to estimate the galaxy merger fraction. The merger label is Class8.6 ("Odd: Merger"). We compute the fraction of test galaxies with predicted merger probability above a threshold.

**Comparison to Lotz et al. 2011:** The $z \approx 0$ major merger rate from observations is $\sim 0.01$–$0.03$ Gyr$^{-1}$ (see their Figure 13, upper right panel). The merger *fraction* (instantaneous fraction of galaxies undergoing a merger) is related to the merger *rate* by the merger observability timescale $T_{\rm obs}$: $f_{\rm merger} = R_{\rm merger} \times T_{\rm obs}$. Typical observability timescales are $\sim 0.5$–$1$ Gyr, so the expected fraction is $\sim 0.5$–$3$%.

**Complicating factors:**
- The GZ2 merger label conflates major and minor mergers, tidal interactions, and close pairs, potentially inflating the fraction.
- Higher-redshift galaxies appear smaller and lower-resolution, making them more likely to be classified as disturbed/merged even when they are not.
- The GZ2 debiasing corrections do not fully account for resolution-dependent classification bias.
- The Lotz et al. rates are in units of Gyr$^{-1}$ (rate per unit time), while we estimate a dimensionless fraction.

In [ ]:
# Load test images
TEST_IMAGE_DIR = Path("test_images")

from ugdatalab.models.galaxy_zoo.images import _preprocess_images

# Get test galaxy IDs from the directory listing
test_files = sorted(TEST_IMAGE_DIR.glob("*.jpg"))
test_galaxy_ids = np.array([int(f.stem) for f in test_files])
print(f"Test galaxies: {len(test_galaxy_ids)}")

# Preprocess test images with same parameters as training
test_images = _preprocess_images(
    TEST_IMAGE_DIR, test_galaxy_ids,
    crop_fraction=0.25, target_size=TARGET_SIZE,
)
print(f"Test images shape: {test_images.shape}")

In [ ]:
# Predict on test set using the best model
# Use dummy labels (zeros) for the Dataset — we only need predictions
dummy_labels = np.zeros((len(test_images), N_LABELS), dtype=np.float32)
test_ds = GalaxyZooDataset(test_images, dummy_labels, transform=None)

test_pred = predict_cnn(best_model, test_ds, DEVICE, batch_size=64)

# Merger fraction
merger_idx = LABEL_COLUMNS.index(MERGER_LABEL)
merger_probs = test_pred[:, merger_idx]

# Report merger fraction at various thresholds
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
print(f"Merger label: {MERGER_LABEL} ({LABEL_DESCRIPTIVE[MERGER_LABEL]})")
print(f"Mean predicted merger probability: {np.mean(merger_probs):.4f}")
print(f"Median predicted merger probability: {np.median(merger_probs):.4f}")
print()
for thresh in thresholds:
    n_above = np.sum(merger_probs > thresh)
    frac = n_above / len(merger_probs)
    print(f"  Threshold > {thresh}: {n_above}/{len(merger_probs)} = {frac*100:.2f}%")

# The mean probability is the most natural estimator of the merger fraction
# since it averages over the continuous label space
merger_fraction = float(np.mean(merger_probs))
print(f"\nEstimated merger fraction (mean prob): {merger_fraction*100:.2f}%")

In [ ]:
# Save evaluation results
np.savez_compressed(
    "evaluation_results.npz",
    val_pred_labels=pred_labels,
    val_true_labels=val_labels,
    test_pred_labels=test_pred,
    test_galaxy_ids=test_galaxy_ids,
    merger_fraction=merger_fraction,
)
print("Saved evaluation_results.npz")